# ESLint — Raw Lint Analysis Output Extraction (JavaScript)

Analyze **JavaScript repositories** with **ESLint** and capture the complete raw lint output required for White Box metric extraction and validation.

This notebook preserves the original ESLint output exactly as emitted by the tool.

**Default benchmark repository:** [javascript-testing-eslint](https://github.com/visvantha-testable/javascript-testing-eslint)

> **Note:** This workflow extracts and parses raw ESLint fields only. It does **not** calculate taxonomy metrics or custom scores.


## Section 1 — Install Dependencies

Install Python packages and verify `node`, `npm`, and `npx eslint`.


In [ ]:
!pip install -q gitpython pandas jupyter
!pip install -q -r requirements.txt


In [ ]:
import os
import sys
from pathlib import Path

os.environ.pop('PYTHONPATH', None)
METRIC_ROOT = Path('.').resolve()
if not (METRIC_ROOT / 'tool' / '_eslint_utils.py').exists():
    METRIC_ROOT = Path('..').resolve()
TOOL_ROOT = METRIC_ROOT / 'tool'
if str(TOOL_ROOT) not in sys.path:
    sys.path.insert(0, str(TOOL_ROOT))

from IPython.display import display

from _eslint_utils import (
    ANALYSIS_TYPE, PROGRAMMING_LANGUAGE, REPO_URL, TOOL_NAME, NotebookLogger,
    collect_prerequisite_versions, ensure_output_dirs, resolve_metric_root,
)

METRIC_ROOT = resolve_metric_root(METRIC_ROOT)
DIRS = ensure_output_dirs(METRIC_ROOT)
OUTPUT_DIR = DIRS['outputs']
WORKSPACE_DIR = DIRS['workspace']
LOGGER = NotebookLogger(OUTPUT_DIR / 'error_log.txt')

PREREQ_DF = collect_prerequisite_versions()
display(PREREQ_DF)


## Section 2 — Configuration

Choose Git clone mode or local repository mode.


In [ ]:
USE_GIT_URL = True

REPO_URL = 'https://github.com/visvantha-testable/javascript-testing-eslint.git'

LOCAL_REPO_PATH = './workspace/javascript-testing-eslint'

WORKSPACE_DIR = './workspace'

OUTPUT_DIR = './outputs'

IF_CLONE_EXISTS = 'reuse'  # reuse | reclone

# Local mode example:
# USE_GIT_URL = False
# LOCAL_REPO_PATH = './workspace/javascript-testing-eslint'


## Section 3 — Repository Setup

Clone or reuse the repository and display repository metadata.


In [ ]:
from IPython.display import display

from _eslint_utils import (
    compute_repository_stats, discover_javascript_files, resolve_repository_path,
)

OUTPUT_PATH = Path(OUTPUT_DIR).resolve()
WORKSPACE_PATH = Path(WORKSPACE_DIR).resolve()
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

try:
    REPO_PATH, CLONE_STATUS = resolve_repository_path(
        USE_GIT_URL, REPO_URL, LOCAL_REPO_PATH, WORKSPACE_PATH, IF_CLONE_EXISTS, LOGGER
    )
except Exception as exc:
    LOGGER.error(f'Repository setup failed: {exc}')
    raise

INVENTORY_DF = discover_javascript_files(REPO_PATH)
REPO_STATS = compute_repository_stats(REPO_PATH, INVENTORY_DF)

print(CLONE_STATUS)
print(f"Repository Name: {REPO_STATS['repository_name']}")
print(f"Repository Size (JS bytes): {REPO_STATS['repository_size_bytes']:,}")
print(f"Number of Directories: {REPO_STATS['directory_count']}")
print(f"Number of JavaScript Files: {REPO_STATS['javascript_file_count']}")
print(f"Current Commit Hash: {REPO_STATS['commit_hash']}")
print(f"package.json present: {REPO_STATS['package_json_present']}")


## Section 4 — Discover JavaScript Files

Recursively discover `*.js`, `*.mjs`, and `*.cjs` files and export the inventory CSV.


In [ ]:
INVENTORY_CSV = OUTPUT_PATH / 'javascript_files_inventory.csv'
INVENTORY_DF.to_csv(INVENTORY_CSV, index=False)
print(f'Total JavaScript Files Found : {REPO_STATS["javascript_file_count"]}')
print(f'Saved inventory to: {INVENTORY_CSV}')
display(INVENTORY_DF.head(20))


## Section 5 — Install Project Dependencies

Run `npm install` and capture stdout, stderr, exit code, and execution time.


In [ ]:
from _eslint_utils import run_npm_install

try:
    NPM_INSTALL_RESULT = run_npm_install(REPO_PATH, LOGGER)
    print('--- npm install stdout ---')
    print(NPM_INSTALL_RESULT['stdout'])
    print('--- npm install stderr ---')
    print(NPM_INSTALL_RESULT['stderr'])
    print(f"Exit Code: {NPM_INSTALL_RESULT['returncode']}")
    print(f"Execution Time (ms): {NPM_INSTALL_RESULT['elapsed_ms']}")
    if not NPM_INSTALL_RESULT['success']:
        LOGGER.error('npm install failed.', file=str(REPO_PATH / 'package.json'))
except Exception as exc:
    NPM_INSTALL_RESULT = {'success': False, 'stdout': '', 'stderr': str(exc), 'returncode': 1, 'elapsed_ms': 0}
    LOGGER.error(f'npm install raised an exception: {exc}')


## Section 6 — Verify ESLint Installation

Verify ESLint is available. If missing, install with `npm install --save-dev eslint`.

If no ESLint configuration exists in the repository, report the issue and stop without modifying source files.


In [ ]:
from _eslint_utils import discover_eslint_config, verify_eslint_installation

ESLINT_CONFIG = discover_eslint_config(REPO_PATH)
ESLINT_VERIFY = {'installed': False, 'version': ''}
if NPM_INSTALL_RESULT.get('success'):
    ESLINT_VERIFY = verify_eslint_installation(REPO_PATH, LOGGER)
    print(f"ESLint version: {ESLINT_VERIFY.get('version', '')}")
    print(f"ESLint installed: {ESLINT_VERIFY.get('installed')}")
else:
    LOGGER.error('Skipped ESLint verification because npm install failed.', file=str(REPO_PATH))

if ESLINT_CONFIG is None:
    LOGGER.error('No ESLint configuration found. Stopping without modifying the repository.', file=str(REPO_PATH))
    print('Supported configs: eslint.config.js, .eslintrc.*, package.json eslintConfig')
else:
    print(f'Using ESLint configuration: {ESLINT_CONFIG}')


## Section 7 — Execute ESLint

Run `npm run lint` when available and `npx eslint . -f json`. Capture stdout, stderr, exit code, and execution duration.


In [ ]:
from _eslint_utils import combine_console_output, has_lint_script, run_eslint_json, run_npm_lint

LINT_SCRIPT_RESULT = None
ESLINT_JSON_RESULT = {
    'stdout': '',
    'stderr': 'Skipped because prerequisites failed.',
    'returncode': 1,
    'elapsed_ms': 0,
    'success': False,
}

if NPM_INSTALL_RESULT.get('success') and ESLINT_CONFIG is not None and ESLINT_VERIFY.get('installed'):
    if has_lint_script(REPO_PATH):
        LINT_SCRIPT_RESULT = run_npm_lint(REPO_PATH, LOGGER)
        print('--- npm run lint stdout ---')
        print(LINT_SCRIPT_RESULT['stdout'])
        print('--- npm run lint stderr ---')
        print(LINT_SCRIPT_RESULT['stderr'])
        print(f"Exit Code: {LINT_SCRIPT_RESULT['returncode']}")
        print(f"Execution Time (ms): {LINT_SCRIPT_RESULT['elapsed_ms']}")
    else:
        print('No npm run lint script found; continuing with npx eslint . -f json')

    ESLINT_JSON_RESULT = run_eslint_json(REPO_PATH, LOGGER)
    print('--- npx eslint . -f json stdout ---')
    print(ESLINT_JSON_RESULT['stdout'][:500] + ('...' if len(ESLINT_JSON_RESULT['stdout']) > 500 else ''))
    print('--- npx eslint . -f json stderr ---')
    print(ESLINT_JSON_RESULT['stderr'])
    print(f"Exit Code: {ESLINT_JSON_RESULT['returncode']}")
    print(f"Execution Time (ms): {ESLINT_JSON_RESULT['elapsed_ms']}")
else:
    LOGGER.error('Skipped ESLint execution because prerequisites were not met.', file=str(REPO_PATH))


## Section 8 — Extract Raw Tool Output

Store the original ESLint JSON output exactly as generated. Do not modify or remove fields.

Raw fields preserved for downstream White Box extraction:

```text
Rule Violations, Rule Severity, Rule Identifier, Violation Count,
File-wise Violations, Line Number, Column Number, Message,
Node Type, Fix Availability, ESLint Summary
```


In [ ]:
RAW_JSON_TEXT = ESLINT_JSON_RESULT.get('stdout', '') or '[]'
RAW_OUTPUT_PATH = OUTPUT_PATH / 'eslint_raw_output.json'
RAW_OUTPUT_PATH.write_text(RAW_JSON_TEXT, encoding='utf-8')

console_chunks = [('npm install', NPM_INSTALL_RESULT)]
if ESLINT_VERIFY.get('install_result'):
    console_chunks.append(('npm install --save-dev eslint', ESLINT_VERIFY['install_result']))
if LINT_SCRIPT_RESULT is not None:
    console_chunks.append(('npm run lint', LINT_SCRIPT_RESULT))
console_chunks.append(('npx eslint . -f json', ESLINT_JSON_RESULT))
CONSOLE_OUTPUT = combine_console_output(console_chunks)

(OUTPUT_PATH / 'eslint_stdout.txt').write_text(ESLINT_JSON_RESULT.get('stdout', ''), encoding='utf-8')
(OUTPUT_PATH / 'eslint_stderr.txt').write_text(ESLINT_JSON_RESULT.get('stderr', ''), encoding='utf-8')
(OUTPUT_PATH / 'eslint_console_output.txt').write_text(CONSOLE_OUTPUT, encoding='utf-8')

print(f'Preserved raw ESLint JSON at: {RAW_OUTPUT_PATH}')


## Section 9 — Parse ESLint Results

Generate violation, file summary, and rule frequency CSV files.


In [ ]:
import json

from _eslint_utils import (
    build_file_summary_dataframe,
    build_rule_frequency_dataframe,
    build_rule_violations_dataframe,
    parse_eslint_json,
)

RECORDS = []
try:
    RECORDS = parse_eslint_json(RAW_JSON_TEXT)
except (json.JSONDecodeError, ValueError) as exc:
    LOGGER.error(f'Malformed ESLint JSON output: {exc}', file=str(RAW_OUTPUT_PATH))

VIOLATIONS_DF = build_rule_violations_dataframe(RECORDS)
FILE_SUMMARY_DF = build_file_summary_dataframe(RECORDS)
RULE_FREQUENCY_DF = build_rule_frequency_dataframe(VIOLATIONS_DF)

VIOLATIONS_CSV = OUTPUT_PATH / 'rule_violations_results.csv'
FILE_SUMMARY_CSV = OUTPUT_PATH / 'file_summary.csv'
RULE_FREQUENCY_CSV = OUTPUT_PATH / 'rule_frequency.csv'
VIOLATIONS_DF.to_csv(VIOLATIONS_CSV, index=False)
FILE_SUMMARY_DF.to_csv(FILE_SUMMARY_CSV, index=False)
RULE_FREQUENCY_DF.to_csv(RULE_FREQUENCY_CSV, index=False)

print(f'Violations CSV: {VIOLATIONS_CSV}')
print(f'File summary CSV: {FILE_SUMMARY_CSV}')
print(f'Rule frequency CSV: {RULE_FREQUENCY_CSV}')
display(VIOLATIONS_DF.head(20))


## Section 10 — Summary Dashboard

Display lint counts and overall ESLint execution status.


In [ ]:
import pandas as pd

from _eslint_utils import build_summary_dashboard, collect_environment_json

EXECUTION_STATUS = 'SUCCESS' if ESLINT_JSON_RESULT.get('returncode') in {0, 1} and RECORDS else 'FAILED'
if ESLINT_CONFIG is None:
    EXECUTION_STATUS = 'MISSING_ESLINT_CONFIG'
SUMMARY = build_summary_dashboard(INVENTORY_DF, VIOLATIONS_DF, RECORDS, EXECUTION_STATUS)
ENVIRONMENT = collect_environment_json(REPO_PATH, str(ESLINT_VERIFY.get('version', '')), ESLINT_CONFIG)

summary_df = pd.DataFrame(
    [
        {'Metric': 'Total JavaScript Files', 'Value': SUMMARY['total_javascript_files']},
        {'Metric': 'Total Violations', 'Value': SUMMARY['total_violations']},
        {'Metric': 'Total Errors', 'Value': SUMMARY['total_errors']},
        {'Metric': 'Total Warnings', 'Value': SUMMARY['total_warnings']},
        {'Metric': 'Fixable Violations', 'Value': SUMMARY['fixable_violations']},
        {'Metric': 'Unique Rules Triggered', 'Value': SUMMARY['unique_rules_triggered']},
        {'Metric': 'ESLint Execution Status', 'Value': SUMMARY['eslint_execution_status']},
    ]
)
display(summary_df)

EXECUTION_METADATA = {
    'clone_status': CLONE_STATUS,
    'repository_stats': REPO_STATS,
    'eslint_config': str(ESLINT_CONFIG) if ESLINT_CONFIG else '',
    'npm_install': NPM_INSTALL_RESULT,
    'eslint_verification': ESLINT_VERIFY,
    'lint_script_result': LINT_SCRIPT_RESULT,
    'eslint_json_result': ESLINT_JSON_RESULT,
    'summary': SUMMARY,
}
(OUTPUT_PATH / 'execution_metadata.json').write_text(json.dumps(EXECUTION_METADATA, indent=2, default=str), encoding='utf-8')
(OUTPUT_PATH / 'environment.json').write_text(json.dumps(ENVIRONMENT, indent=2), encoding='utf-8')


## Expected Raw Output Support

The preserved ESLint JSON supports later extraction of White Box metrics such as Lint / Rule Violations, Violation Density per KLOC, Unused Variable Detection, Naming Convention Validation, Code Style Rule Validation, Complexity Rule Detection, Rule Severity Classification, Multiple Violations Detection, Configuration File Handling, CI/CD Integration Validation, and Violation Reporting Validation.

This notebook does **not** calculate those metrics.


## Section 11 — Error Handling

Review captured errors and confirm deliverables under `outputs/`.


In [ ]:
from _eslint_utils import read_text

LOGGER.write_errors()
ERROR_LOG = OUTPUT_PATH / 'error_log.txt'
print(f'Error log: {ERROR_LOG}')
print(read_text(ERROR_LOG) or 'No errors recorded.')

deliverables = [
    'eslint_raw_output.json',
    'eslint_stdout.txt',
    'eslint_stderr.txt',
    'eslint_console_output.txt',
    'rule_violations_results.csv',
    'file_summary.csv',
    'rule_frequency.csv',
    'javascript_files_inventory.csv',
    'execution_metadata.json',
    'environment.json',
    'error_log.txt',
]
print('\nDeliverables:')
for name in deliverables:
    path = OUTPUT_PATH / name
    status = 'OK' if path.exists() else 'MISSING'
    print(f'  [{status}] {path}')
